# Chapter 2 — Tokenization: From Raw Text to Model Inputs

This notebook explains the bridge between human text and transformer tensors. It starts from very small examples—character tokenization, integer IDs, and one-hot vectors—then moves to real subword tokenization with the DistilBERT tokenizer. This progression is useful because it shows why modern NLP does not feed raw strings directly into neural networks.

The central concept is that a tokenizer converts text into model-readable fields such as `input_ids` and `attention_mask`. `input_ids` are vocabulary indices, while `attention_mask` tells the model which positions are real tokens and which positions are padding. These are the exact fields that will be passed to DistilBERT in the modeling notebook.

The outputs are important here. The printed tokens, IDs, tokenizer vocabulary size, model maximum length, and encoded dataset columns show the concrete data structures that the model receives.

## Imports

This notebook uses PyTorch to demonstrate one-hot vectors, Hugging Face Datasets to load the Emotion data, Pandas for a tiny categorical example, and Transformers for modern tokenizers.

In [2]:
# Imports for tensor examples, dataset loading, tabular display, and tokenizer utilities.
import torch
import torch.nn.functional as F
from datasets import load_dataset
import pandas as pd
import transformers

## Loading the Emotion dataset

The same dataset is loaded again so tokenization can be demonstrated independently from the EDA notebook.

In [3]:
# Load the Emotion dataset for tokenizer experiments.
emotions = load_dataset('emotion')

Using the latest cached version of the dataset since emotion couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'split' at C:\Users\98922\.cache\huggingface\datasets\emotion\split\0.0.0\cab853a1dbdf4c42c2b3ef2173804746df8825fe (last modified on Fri Jan  2 23:41:12 2026).


## Character-level tokenization

The simplest tokenizer is `list(text)`, which splits a string into characters. This is intentionally primitive, but it helps introduce the idea of discrete tokens.

In [4]:
# Demonstrate character-level tokenization with a simple Python list conversion.
text = "Tokenizing text is a core task of NLP."
char_tokenized_text = list(text)
print(char_tokenized_text)

['T', 'o', 'k', 'e', 'n', 'i', 'z', 'i', 'n', 'g', ' ', 't', 'e', 'x', 't', ' ', 'i', 's', ' ', 'a', ' ', 'c', 'o', 'r', 'e', ' ', 't', 'a', 's', 'k', ' ', 'o', 'f', ' ', 'N', 'L', 'P', '.']


## Building a character vocabulary

A vocabulary maps each unique token to an integer. Neural networks do not consume raw characters or words; they consume integer IDs and tensors derived from those IDs.

In [5]:
# Build a tiny vocabulary that maps each unique character to an integer ID.
token2idx = {char:idx for idx, char in enumerate(sorted(set(char_tokenized_text)))}
print(token2idx)

{' ': 0, '.': 1, 'L': 2, 'N': 3, 'P': 4, 'T': 5, 'a': 6, 'c': 7, 'e': 8, 'f': 9, 'g': 10, 'i': 11, 'k': 12, 'n': 13, 'o': 14, 'r': 15, 's': 16, 't': 17, 'x': 18, 'z': 19}


## Converting tokens to input IDs

This cell replaces each character with its vocabulary index. The resulting list is the simplest version of the `input_ids` concept used by transformer tokenizers.

In [6]:
# Convert each character token into its integer vocabulary ID.
input_ids = [token2idx[token] for token in char_tokenized_text]
print(input_ids)

[5, 14, 12, 8, 13, 11, 19, 11, 13, 10, 0, 17, 8, 18, 17, 0, 11, 16, 0, 6, 0, 7, 14, 15, 8, 0, 17, 6, 16, 12, 0, 14, 9, 0, 3, 2, 4, 1]


## A small categorical encoding example

The DataFrame shows a familiar tabular analogy: categories can be represented with numeric labels. This prepares the intuition for one-hot encoding.

In [7]:
# Create a tiny categorical DataFrame to illustrate label-style encoding.
categorical_df = pd.DataFrame(
    {"Name":["Bumblebee", "Optimus Prime", "Megatron"], "Label ID":[0, 1, 2]}
)
categorical_df

,Name,Label ID
0,Bumblebee,0
1,Optimus Prime,1
2,Megatron,2


## One-hot encoding categories

One-hot encoding expands each category into a sparse vector. This is easy to understand but inefficient for large vocabularies, which is one reason modern NLP uses embeddings instead.

In [8]:
# Use Pandas one-hot encoding to represent categories as indicator columns.
pd.get_dummies(categorical_df["Name"])

,Bumblebee,Megatron,Optimus Prime
0,True,False,False
1,False,False,True
2,False,True,False


## One-hot encoding token IDs with PyTorch

Here the character IDs are converted into one-hot vectors. The output shape shows that every token position becomes a vector with one dimension per vocabulary item.

In [9]:
# Convert token IDs into one-hot vectors with PyTorch.
input_ids = torch.tensor(input_ids)
one_hot_encodings = F.one_hot(input_ids, num_classes=len(token2idx))
print(one_hot_encodings.size())

torch.Size([38, 20])


## Inspecting one token representation

This cell connects the original token, its integer ID, and its one-hot vector. It makes the abstraction concrete before moving to real tokenizers.

In [10]:
# Inspect the connection between one raw token, its integer ID, and its one-hot vector.
print(f"Token: {char_tokenized_text[0]}")
print(f"Tensor index: {input_ids[0]}")
print(f"One Hot Encoded token: {one_hot_encodings[0]}")

Token: T
Tensor index: 5
One Hot Encoded token: tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


## Naive word tokenization

Splitting on whitespace is a simple word tokenizer. It is useful for demonstration, but it fails on punctuation, unknown words, casing, and subword structure.

In [11]:
# Demonstrate naive whitespace word tokenization.
word_tokenized_text = text.split()
print(word_tokenized_text)

['Tokenizing', 'text', 'is', 'a', 'core', 'task', 'of', 'NLP.']


## Loading pretrained DistilBERT tokenizers

`AutoTokenizer` loads the correct tokenizer class from the checkpoint name. The explicit `DistilBertTokenizer` is loaded too, showing that both interfaces can point to the same tokenizer behavior.

In [12]:
# Load tokenizer objects for the DistilBERT checkpoint.
# AutoTokenizer is preferred because it selects the correct tokenizer class automatically.
from transformers import AutoTokenizer
from transformers import DistilBertTokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(model_ckpt)

## Encoding text with two tokenizer interfaces

Both tokenizers convert the same sentence into model inputs. The output includes token IDs and attention masks, which are the core inputs expected by DistilBERT.

In [14]:
# Encode the same text with both tokenizer interfaces and compare the returned fields.
encoded_text1 = tokenizer(text)
encoded_text2 = distilbert_tokenizer(text)
print(encoded_text1, '\n', encoded_text2)

{'input_ids': [101, 19204, 6026, 3793, 2003, 1037, 4563, 4708, 1997, 17953, 2361, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]} 
 {'input_ids': [101, 19204, 6026, 3793, 2003, 1037, 4563, 4708, 1997, 17953, 2361, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Converting IDs back to tokens

This reveals DistilBERT's WordPiece-style subword tokens, including special tokens such as `[CLS]` and `[SEP]`.

In [15]:
# Convert token IDs back to tokenizer tokens for inspection.
tokens = tokenizer.convert_ids_to_tokens(encoded_text1.input_ids)
print(tokens)

['[CLS]', 'token', '##izing', 'text', 'is', 'a', 'core', 'task', 'of', 'nl', '##p', '.', '[SEP]']


## Reconstructing text from tokens

Converting tokens back to a string is useful for debugging tokenization and understanding how special/subword tokens are handled.

In [16]:
# Reconstruct a readable string from tokenizer tokens.
print(tokenizer.convert_tokens_to_string(tokens))

[CLS] tokenizing text is a core task of nlp. [SEP]


## Inspecting tokenizer metadata

The vocabulary size, maximum sequence length, and expected input names define the tokenizer-model contract. These values guide padding, truncation, and batch construction.

In [17]:
# Print tokenizer metadata that defines the model input contract.
print(f"Vocab Size of DistilBert: {tokenizer.vocab_size}")
print(f"Max Length of Model Input: {tokenizer.model_max_length}")
print(f"Model Expected Inputs: {tokenizer.model_input_names} ")

Vocab Size of DistilBert: 30522
Max Length of Model Input: 512
Model Expected Inputs: ['input_ids', 'attention_mask'] 


## Writing a batched tokenization function

This function will be passed to `Dataset.map()`. It tokenizes the `text` column and pads/truncates examples into a consistent model-ready format.

In [18]:
# Define a reusable preprocessing function for Hugging Face Dataset.map().
def tokenize(batch):
    return (tokenizer(batch["text"], padding=True, truncation=True))

## Testing tokenization on two examples

Before mapping over the whole dataset, it is good practice to test the preprocessing function on a tiny batch and inspect the returned fields.

In [19]:
# Test the tokenization function on a tiny batch before processing the whole dataset.
print(tokenize(emotions["train"][:2]))

{'input_ids': [[101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998, 2003, 8300, 102]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


## Tokenizing the full dataset

`map()` applies the tokenization function to every split. With `batched=True`, the tokenizer processes batches efficiently instead of one row at a time.

In [20]:
# Apply tokenization to every split in the Emotion dataset.
emotions_encoded = emotions.map(tokenize, batched=True, batch_size=None)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## Checking the new encoded columns

After tokenization, the dataset contains extra fields such as `input_ids` and `attention_mask`. These are the columns used by the transformer model in the next notebook.

In [21]:
# Verify that encoded model-input columns were added to the dataset.
print(emotions_encoded["train"].column_names)

['text', 'label', 'input_ids', 'attention_mask']


## Empty working cell

This original empty cell is preserved for experiments, such as trying a different checkpoint or inspecting token lengths.

In [ ]:
# Empty scratch cell preserved from the original notebook.
